# Full Snapshot Smoke Tests

Validates the snapshot after export by checking:
1. ID format correctness for all entities
2. Entity counts vs OpenAlex API
3. Works field completeness vs API response schema
4. Sample works field-by-field comparison against API
5. Authorships with affiliations at expected rates

In [ ]:
import json
import re
import requests
import time
from pyspark.sql import functions as F

API_BASE = "https://api.openalex.org"
POLITE_EMAIL = "casey@ourresearch.org"

results = []

def record(test_name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"test": test_name, "passed": passed, "detail": detail})
    icon = "OK" if passed else "XX"
    print(f"  [{icon}] {test_name}: {detail}")

def api_get(path, params=None):
    """Call OpenAlex API with polite pool and retry."""
    url = f"{API_BASE}/{path}"
    p = {"mailto": POLITE_EMAIL}
    if params:
        p.update(params)
    for attempt in range(3):
        try:
            r = requests.get(url, params=p, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                raise

print("Setup complete.")

## Test 1: ID Format Validation
Every entity must have IDs matching the expected OpenAlex URL pattern.

In [ ]:
print("=" * 60)
print("TEST 1: ID Format Validation")
print("=" * 60)

ENTITY_ID_SPECS = [
    ("works",        "openalex.works.openalex_works_snapshot",              r"^https://openalex\.org/W\d+$"),
    ("authors",      "openalex.authors.openalex_authors_snapshot",          r"^https://openalex\.org/A\d+$"),
    ("institutions", "openalex.institutions.openalex_institutions_snapshot", r"^https://openalex\.org/I\d+$"),
    ("sources",      "openalex.sources.openalex_sources_snapshot",          r"^https://openalex\.org/S\d+$"),
    ("publishers",   "openalex.publishers.openalex_publishers_snapshot",    r"^https://openalex\.org/P\d+$"),
    ("funders",      "openalex.funders.openalex_funders_snapshot",          r"^https://openalex\.org/F\d+$"),
    ("topics",       "openalex.common.openalex_topics_snapshot",            r"^https://openalex\.org/T\d+$"),
    ("concepts",     "openalex.common.openalex_concepts_snapshot",          r"^https://openalex\.org/C\d+$"),
    ("awards",       "openalex.awards.openalex_awards_snapshot",            r"^https://openalex\.org/G\d+$"),
    ("keywords",     "openalex.common.openalex_keywords_snapshot",          r"^https://openalex\.org/keywords/.+$"),
]

for entity_name, table, pattern in ENTITY_ID_SPECS:
    df = spark.read.table(table)
    total = df.count()
    bad_ids = df.filter(~F.col("id").rlike(pattern)).count()
    null_ids = df.filter(F.col("id").isNull()).count()
    passed = (bad_ids == 0) and (null_ids == 0)
    detail = f"{total:,} records, {null_ids} null IDs, {bad_ids} malformed IDs (pattern: {pattern})"
    record(f"id_format_{entity_name}", passed, detail)

# Subfields, fields, domains use openalex.org URLs with the hierarchy prefix
HIERARCHY_SPECS = [
    ("subfields", "openalex.common.openalex_subfields_snapshot", r"^https://openalex\.org/subfields/\d+$"),
    ("fields",    "openalex.common.openalex_fields_snapshot",    r"^https://openalex\.org/fields/\d+$"),
    ("domains",   "openalex.common.openalex_domains_snapshot",   r"^https://openalex\.org/domains/\d+$"),
]

for entity_name, table, pattern in HIERARCHY_SPECS:
    df = spark.read.table(table)
    total = df.count()
    bad_ids = df.filter(~F.col("id").rlike(pattern)).count()
    null_ids = df.filter(F.col("id").isNull()).count()
    passed = (bad_ids == 0) and (null_ids == 0)
    detail = f"{total:,} records, {null_ids} null IDs, {bad_ids} malformed IDs (pattern: {pattern})"
    record(f"id_format_{entity_name}", passed, detail)

## Test 2: Entity Counts vs API
Snapshot record counts should be within 5% of the live API counts.

In [ ]:
print("=" * 60)
print("TEST 2: Entity Counts vs API")
print("=" * 60)

COUNT_TOLERANCE = 0.05  # 5% tolerance

ENTITY_COUNT_SPECS = [
    ("works",        "openalex.works.openalex_works_snapshot",              "works"),
    ("authors",      "openalex.authors.openalex_authors_snapshot",          "authors"),
    ("institutions", "openalex.institutions.openalex_institutions_snapshot", "institutions"),
    ("sources",      "openalex.sources.openalex_sources_snapshot",          "sources"),
    ("publishers",   "openalex.publishers.openalex_publishers_snapshot",    "publishers"),
    ("funders",      "openalex.funders.openalex_funders_snapshot",          "funders"),
    ("topics",       "openalex.common.openalex_topics_snapshot",            "topics"),
    ("concepts",     "openalex.common.openalex_concepts_snapshot",          "concepts"),
    ("keywords",     "openalex.common.openalex_keywords_snapshot",          "keywords"),
]

for entity_name, table, api_endpoint in ENTITY_COUNT_SPECS:
    snapshot_count = spark.read.table(table).count()
    api_resp = api_get(api_endpoint, {"per_page": "1", "select": "id"})
    api_count = api_resp["meta"]["count"]
    
    if api_count == 0:
        pct_diff = 0 if snapshot_count == 0 else float('inf')
    else:
        pct_diff = abs(snapshot_count - api_count) / api_count
    
    passed = pct_diff <= COUNT_TOLERANCE
    detail = f"snapshot={snapshot_count:,}  api={api_count:,}  diff={pct_diff:.2%}"
    record(f"count_{entity_name}", passed, detail)

## Test 3: Works Field Completeness vs API
The snapshot works table should have all the fields present in API responses.
Some fields are intentionally excluded from the snapshot (e.g. `content_urls`).

In [ ]:
print("=" * 60)
print("TEST 3: Works Field Completeness vs API")
print("=" * 60)

# Fetch a well-populated work from the API to get the canonical field list
api_work = api_get("works/W2741809807")
api_fields = set(api_work.keys())

# Get snapshot table fields
snapshot_fields = set(spark.read.table("openalex.works.openalex_works_snapshot").columns)

# Fields intentionally not in snapshot (internal API-only fields)
KNOWN_EXCLUSIONS = {"content_urls"}

# Fields in snapshot but not in API (snapshot extras)
KNOWN_EXTRAS = {"authors_count", "has_abstract"}

missing_from_snapshot = api_fields - snapshot_fields - KNOWN_EXCLUSIONS
extra_in_snapshot = snapshot_fields - api_fields - KNOWN_EXTRAS

passed = len(missing_from_snapshot) == 0
detail = f"API has {len(api_fields)} fields, snapshot has {len(snapshot_fields)} fields"
if missing_from_snapshot:
    detail += f"\n    MISSING from snapshot: {sorted(missing_from_snapshot)}"
if extra_in_snapshot:
    detail += f"\n    EXTRA in snapshot (not in API): {sorted(extra_in_snapshot)}"
if KNOWN_EXCLUSIONS & api_fields:
    detail += f"\n    Known exclusions (OK): {sorted(KNOWN_EXCLUSIONS & api_fields)}"
record("works_field_completeness", passed, detail)

print(f"\n  API fields:      {sorted(api_fields)}")
print(f"  Snapshot fields: {sorted(snapshot_fields)}")

## Test 4: Sample Works Comparison Against API
Fetch 5 works from the API and verify key fields match the snapshot table.

In [ ]:
print("=" * 60)
print("TEST 4: Sample Works Comparison Against API")
print("=" * 60)

# Get 5 diverse works from the API: a highly-cited one, a recent one, and random ones
sample_work_ids = [
    "W2741809807",   # well-known highly cited work
    "W2741809807",   # placeholder - will be replaced by random sample
]

# Get a few random works from the snapshot to cross-check
random_works = (
    spark.read.table("openalex.works.openalex_works_snapshot")
    .filter(F.col("doi").isNotNull())
    .filter(F.col("publication_year") >= 2020)
    .filter(F.size("authorships") > 0)
    .filter(F.col("primary_topic").isNotNull())
    .orderBy(F.rand(seed=42))
    .limit(5)
    .select("id")
    .collect()
)

sample_ids = [row["id"] for row in random_works]
print(f"  Testing {len(sample_ids)} sample works: {sample_ids}")

# Fields to compare (scalar and simple fields)
SCALAR_FIELDS = [
    "id", "doi", "title", "display_name", "publication_year",
    "language", "type", "is_retracted", "is_paratext", "is_xpac",
    "cited_by_count", "locations_count", "has_fulltext",
    "countries_distinct_count", "institutions_distinct_count",
    "referenced_works_count",
]

# Fields to compare structurally (check key sub-fields)
STRUCT_CHECKS = [
    "open_access",
    "biblio",
    "primary_topic",
    "citation_normalized_percentile",
]

all_match = True
for work_url_id in sample_ids:
    # Extract the short ID (W12345) from full URL
    short_id = work_url_id.split("/")[-1]
    
    # Get from API
    try:
        api_work = api_get(f"works/{short_id}")
    except Exception as e:
        record(f"sample_work_{short_id}", False, f"API fetch failed: {e}")
        all_match = False
        continue
    
    # Get from snapshot
    snap_row = (
        spark.read.table("openalex.works.openalex_works_snapshot")
        .filter(F.col("id") == work_url_id)
        .first()
    )
    
    if snap_row is None:
        record(f"sample_work_{short_id}", False, "Not found in snapshot table")
        all_match = False
        continue
    
    snap_dict = snap_row.asDict(recursive=True)
    
    mismatches = []
    for field in SCALAR_FIELDS:
        api_val = api_work.get(field)
        snap_val = snap_dict.get(field)
        
        # Normalize: treat None and missing the same
        if api_val is None and snap_val is None:
            continue
        
        # publication_date is a date object in snapshot vs string in API
        if field == "publication_date":
            snap_val = str(snap_val) if snap_val else None
        
        if str(api_val) != str(snap_val):
            mismatches.append(f"{field}: api={api_val!r} snap={snap_val!r}")
    
    # Check open_access sub-fields
    if "open_access" in api_work and "open_access" in snap_dict:
        api_oa = api_work["open_access"]
        snap_oa = snap_dict["open_access"]
        for oa_field in ["is_oa", "oa_status"]:
            if api_oa.get(oa_field) != snap_oa.get(oa_field):
                mismatches.append(f"open_access.{oa_field}: api={api_oa.get(oa_field)!r} snap={snap_oa.get(oa_field)!r}")
    
    # Check primary_topic
    if api_work.get("primary_topic") and snap_dict.get("primary_topic"):
        api_pt = api_work["primary_topic"]
        snap_pt = snap_dict["primary_topic"]
        if api_pt.get("id") != snap_pt.get("id"):
            mismatches.append(f"primary_topic.id: api={api_pt.get('id')!r} snap={snap_pt.get('id')!r}")
    
    # Check authorships count
    api_auth_count = len(api_work.get("authorships", []))
    snap_auth_count = len(snap_dict.get("authorships", []))
    if api_auth_count != snap_auth_count:
        mismatches.append(f"authorships_count: api={api_auth_count} snap={snap_auth_count}")
    
    # Check topics count
    api_topics_count = len(api_work.get("topics", []))
    snap_topics_count = len(snap_dict.get("topics", []))
    if api_topics_count != snap_topics_count:
        mismatches.append(f"topics_count: api={api_topics_count} snap={snap_topics_count}")
    
    # Check referenced_works count
    api_ref_count = len(api_work.get("referenced_works", []))
    snap_ref_count = len(snap_dict.get("referenced_works", []))
    if api_ref_count != snap_ref_count:
        mismatches.append(f"referenced_works_len: api={api_ref_count} snap={snap_ref_count}")
    
    # Check concepts count
    api_concepts_count = len(api_work.get("concepts", []))
    snap_concepts_count = len(snap_dict.get("concepts", []))
    if api_concepts_count != snap_concepts_count:
        mismatches.append(f"concepts_count: api={api_concepts_count} snap={snap_concepts_count}")
    
    # Check locations count
    api_loc_count = len(api_work.get("locations", []))
    snap_loc_count = len(snap_dict.get("locations", []))
    if api_loc_count != snap_loc_count:
        mismatches.append(f"locations_len: api={api_loc_count} snap={snap_loc_count}")
    
    passed = len(mismatches) == 0
    if not passed:
        all_match = False
    detail = f"{len(mismatches)} mismatches" + (f": {'; '.join(mismatches[:5])}" if mismatches else "")
    record(f"sample_work_{short_id}", passed, detail)
    time.sleep(0.2)  # polite rate limiting

## Test 5: Authorships with Affiliations
Thresholds derived from actual data (stable across 1-week Delta time travel):
- All works: ~89.9% have authorships, ~32% have institutional affiliations
- 2020+ works: ~95.7% have authorships, ~49.5% have affiliations
- 100% of affiliated works have raw_affiliation_strings

In [ ]:
print("=" * 60)
print("TEST 5: Authorships with Affiliations")
print("=" * 60)

works_df = spark.read.table("openalex.works.openalex_works_snapshot")

# Overall stats
total_works = works_df.count()
works_with_authorships = works_df.filter(F.size("authorships") > 0).count()

# Works where at least one authorship has at least one institution
works_with_affiliations = works_df.filter(
    F.exists("authorships", lambda a: F.size(a.institutions) > 0)
).count()

auth_pct = works_with_authorships / total_works if total_works > 0 else 0
affil_pct = works_with_affiliations / total_works if total_works > 0 else 0

# Actual rate is ~89.9%, threshold at 85% gives safe margin
passed_auth = auth_pct >= 0.85
record("works_have_authorships", passed_auth,
       f"{works_with_authorships:,}/{total_works:,} ({auth_pct:.1%}) have authorships [threshold: 85%]")

# Actual rate is ~32%, threshold at 28% gives safe margin
passed_affil = affil_pct >= 0.28
record("works_have_affiliations", passed_affil,
       f"{works_with_affiliations:,}/{total_works:,} ({affil_pct:.1%}) have institutional affiliations [threshold: 28%]")

# For recent works (2020+), actual affiliation rate is ~49.5%
# Threshold at 44% gives safe margin
recent_works = works_df.filter(F.col("publication_year") >= 2020)
recent_total = recent_works.count()
recent_with_affil = recent_works.filter(
    F.exists("authorships", lambda a: F.size(a.institutions) > 0)
).count()
recent_pct = recent_with_affil / recent_total if recent_total > 0 else 0

passed_recent = recent_pct >= 0.44
record("recent_works_affiliations", passed_recent,
       f"{recent_with_affil:,}/{recent_total:,} ({recent_pct:.1%}) of 2020+ works have affiliations [threshold: 44%]")

# Actual rate is ~100% - affiliated works nearly always have raw_affiliation_strings
# Threshold at 95% to catch catastrophic breakage
works_sample = (
    works_df
    .filter(F.exists("authorships", lambda a: F.size(a.institutions) > 0))
    .limit(10000)
)
with_ras = works_sample.filter(
    F.exists("authorships", lambda a: F.size(a.raw_affiliation_strings) > 0)
).count()
sample_total = works_sample.count()
ras_pct = with_ras / sample_total if sample_total > 0 else 0

passed_ras = ras_pct >= 0.95
record("raw_affiliation_strings_populated", passed_ras,
       f"{with_ras:,}/{sample_total:,} ({ras_pct:.1%}) of affiliated works have raw_affiliation_strings [threshold: 95%]")

## Test 5b: Authorships with Affiliations (non-xpac only)
Filtering out x-pac works gives a cleaner signal on affiliation coverage.
Thresholds derived from actual data (stable across 1-week Delta time travel):
- Non-xpac works: ~92.4% have authorships, ~40.3% have affiliations
- Non-xpac 2020+: ~93.5% have authorships, ~56.1% have affiliations, ~61% have raw_affiliation_strings

In [ ]:
print("=" * 60)
print("TEST 5b: Authorships with Affiliations (non-xpac)")
print("=" * 60)

works_df = spark.read.table("openalex.works.openalex_works_snapshot")
non_xpac = works_df.filter(F.col("is_xpac") == False)

# Non-xpac overall stats
total_non_xpac = non_xpac.count()
non_xpac_with_auth = non_xpac.filter(F.size("authorships") > 0).count()
non_xpac_with_affil = non_xpac.filter(
    F.exists("authorships", lambda a: F.size(a.institutions) > 0)
).count()

auth_pct = non_xpac_with_auth / total_non_xpac if total_non_xpac > 0 else 0
affil_pct = non_xpac_with_affil / total_non_xpac if total_non_xpac > 0 else 0

# Actual rate is ~92.4%, threshold at 88%
passed_auth = auth_pct >= 0.88
record("non_xpac_have_authorships", passed_auth,
       f"{non_xpac_with_auth:,}/{total_non_xpac:,} ({auth_pct:.1%}) non-xpac works have authorships [threshold: 88%]")

# Actual rate is ~40.3%, threshold at 36%
passed_affil = affil_pct >= 0.36
record("non_xpac_have_affiliations", passed_affil,
       f"{non_xpac_with_affil:,}/{total_non_xpac:,} ({affil_pct:.1%}) non-xpac works have affiliations [threshold: 36%]")

# Non-xpac 2020+ works
recent_non_xpac = non_xpac.filter(F.col("publication_year") >= 2020)
recent_total = recent_non_xpac.count()
recent_with_auth = recent_non_xpac.filter(F.size("authorships") > 0).count()
recent_with_affil = recent_non_xpac.filter(
    F.exists("authorships", lambda a: F.size(a.institutions) > 0)
).count()
recent_with_ras = recent_non_xpac.filter(
    F.exists("authorships", lambda a: F.size(a.raw_affiliation_strings) > 0)
).count()

recent_auth_pct = recent_with_auth / recent_total if recent_total > 0 else 0
recent_affil_pct = recent_with_affil / recent_total if recent_total > 0 else 0
recent_ras_pct = recent_with_ras / recent_total if recent_total > 0 else 0

# Actual rate is ~93.5%, threshold at 89%
passed_recent_auth = recent_auth_pct >= 0.89
record("non_xpac_2020_authorships", passed_recent_auth,
       f"{recent_with_auth:,}/{recent_total:,} ({recent_auth_pct:.1%}) non-xpac 2020+ have authorships [threshold: 89%]")

# Actual rate is ~56.1%, threshold at 50%
passed_recent_affil = recent_affil_pct >= 0.50
record("non_xpac_2020_affiliations", passed_recent_affil,
       f"{recent_with_affil:,}/{recent_total:,} ({recent_affil_pct:.1%}) non-xpac 2020+ have affiliations [threshold: 50%]")

# Actual rate is ~61%, threshold at 55%
passed_recent_ras = recent_ras_pct >= 0.55
record("non_xpac_2020_raw_affil_strings", passed_recent_ras,
       f"{recent_with_ras:,}/{recent_total:,} ({recent_ras_pct:.1%}) non-xpac 2020+ have raw_affiliation_strings [threshold: 55%]")

## Test 6: Works Nested Structure Spot Checks
Verify nested structures (authorships, locations, topics) have the expected sub-fields.

In [ ]:
print("=" * 60)
print("TEST 6: Works Nested Structure Spot Checks")
print("=" * 60)

works_df = spark.read.table("openalex.works.openalex_works_snapshot")

# Get a work with rich data for structure checking
rich_work = (
    works_df
    .filter(F.size("authorships") > 2)
    .filter(F.size("locations") > 0)
    .filter(F.col("primary_topic").isNotNull())
    .filter(F.size("concepts") > 0)
    .limit(1)
    .first()
)

if rich_work:
    rd = rich_work.asDict(recursive=True)
    
    # Check authorships structure
    expected_auth_fields = {"author", "author_position", "affiliations", "countries",
                           "raw_author_name", "is_corresponding", "raw_affiliation_strings", "institutions"}
    actual_auth_fields = set(rd["authorships"][0].keys()) if rd.get("authorships") else set()
    auth_ok = expected_auth_fields.issubset(actual_auth_fields)
    record("authorships_structure", auth_ok,
           f"expected={sorted(expected_auth_fields)}, actual={sorted(actual_auth_fields)}")
    
    # Check locations structure
    expected_loc_fields = {"source", "is_oa", "landing_page_url", "pdf_url",
                          "license", "license_id", "version", "is_accepted", "is_published"}
    actual_loc_fields = set(rd["locations"][0].keys()) if rd.get("locations") else set()
    loc_ok = expected_loc_fields.issubset(actual_loc_fields)
    record("locations_structure", loc_ok,
           f"expected={sorted(expected_loc_fields)}, actual={sorted(actual_loc_fields)}")
    
    # Check primary_topic structure
    expected_topic_fields = {"id", "display_name", "score", "subfield", "field", "domain"}
    actual_topic_fields = set(rd["primary_topic"].keys()) if rd.get("primary_topic") else set()
    topic_ok = expected_topic_fields.issubset(actual_topic_fields)
    record("primary_topic_structure", topic_ok,
           f"expected={sorted(expected_topic_fields)}, actual={sorted(actual_topic_fields)}")
    
    # Check concepts structure
    expected_concept_fields = {"id", "wikidata", "display_name", "level", "score"}
    actual_concept_fields = set(rd["concepts"][0].keys()) if rd.get("concepts") else set()
    concept_ok = expected_concept_fields.issubset(actual_concept_fields)
    record("concepts_structure", concept_ok,
           f"expected={sorted(expected_concept_fields)}, actual={sorted(actual_concept_fields)}")
    
    # Check open_access structure
    expected_oa_fields = {"is_oa", "oa_status", "oa_url", "any_repository_has_fulltext"}
    actual_oa_fields = set(rd["open_access"].keys()) if rd.get("open_access") else set()
    oa_ok = expected_oa_fields.issubset(actual_oa_fields)
    record("open_access_structure", oa_ok,
           f"expected={sorted(expected_oa_fields)}, actual={sorted(actual_oa_fields)}")
    
    # Check concept IDs have the right prefix
    concept_ids = [c["id"] for c in rd.get("concepts", []) if c.get("id")]
    all_c_ids_ok = all(re.match(r"^https://openalex\.org/C\d+$", cid) for cid in concept_ids)
    record("concepts_id_prefix", all_c_ids_ok,
           f"Checked {len(concept_ids)} concept IDs in work")
    
    # Check referenced_works have the right prefix
    ref_works = rd.get("referenced_works", [])
    if ref_works:
        all_ref_ok = all(re.match(r"^https://openalex\.org/W\d+$", rw) for rw in ref_works[:20])
        record("referenced_works_prefix", all_ref_ok,
               f"Checked {min(len(ref_works), 20)} referenced_works IDs")
    else:
        record("referenced_works_prefix", True, "No referenced works (OK)")
else:
    record("nested_structure_checks", False, "Could not find a rich work for testing")

## Test 7: Null Checks on Required Fields
Certain fields should never be null across all works.

In [ ]:
print("=" * 60)
print("TEST 7: Null Checks on Required Fields")
print("=" * 60)

works_df = spark.read.table("openalex.works.openalex_works_snapshot")
total_works = works_df.count()

# Fields that should never be null
REQUIRED_FIELDS = ["id", "type", "is_retracted", "is_paratext"]

for field in REQUIRED_FIELDS:
    null_count = works_df.filter(F.col(field).isNull()).count()
    passed = null_count == 0
    record(f"not_null_{field}", passed,
           f"{null_count:,}/{total_works:,} null values")

# Fields that should be mostly non-null (>90%)
MOSTLY_REQUIRED = ["title", "publication_year", "updated_date", "created_date"]

for field in MOSTLY_REQUIRED:
    null_count = works_df.filter(F.col(field).isNull()).count()
    null_pct = null_count / total_works if total_works > 0 else 0
    passed = null_pct < 0.10
    record(f"mostly_not_null_{field}", passed,
           f"{null_count:,}/{total_works:,} ({null_pct:.2%}) null values")

## Test 8: Duplicate ID Check
No entity should have duplicate IDs.

In [ ]:
print("=" * 60)
print("TEST 8: Duplicate ID Check")
print("=" * 60)

DUP_CHECK_TABLES = [
    ("works",        "openalex.works.openalex_works_snapshot"),
    ("authors",      "openalex.authors.openalex_authors_snapshot"),
    ("institutions", "openalex.institutions.openalex_institutions_snapshot"),
    ("sources",      "openalex.sources.openalex_sources_snapshot"),
    ("publishers",   "openalex.publishers.openalex_publishers_snapshot"),
    ("funders",      "openalex.funders.openalex_funders_snapshot"),
    ("topics",       "openalex.common.openalex_topics_snapshot"),
    ("concepts",     "openalex.common.openalex_concepts_snapshot"),
    ("keywords",     "openalex.common.openalex_keywords_snapshot"),
]

for entity_name, table in DUP_CHECK_TABLES:
    df = spark.read.table(table)
    total = df.count()
    distinct = df.select("id").distinct().count()
    dups = total - distinct
    passed = dups == 0
    record(f"no_duplicates_{entity_name}", passed,
           f"{total:,} total, {distinct:,} distinct, {dups:,} duplicates")

## Test 9: S3 Manifest Validation
Verify that manifests were written to S3 for every entity and that their record counts
match the snapshot tables. This confirms the actual S3 export completed correctly.

In [ ]:
print("=" * 60)
print("TEST 9: S3 Manifest Validation")
print("=" * 60)

from datetime import datetime as _dt
date_str = _dt.now().strftime("%Y-%m-%d")
s3_base = f"s3://openalex-snapshots/full/{date_str}"

# All entities that should have manifests after a full snapshot export
MANIFEST_ENTITIES = [
    # (entity_name, snapshot_table_or_None)
    # snapshot_table is used to cross-check record counts where available
    ("works",              "openalex.works.openalex_works_snapshot"),
    ("authors",            "openalex.authors.openalex_authors_snapshot"),
    ("institutions",       "openalex.institutions.openalex_institutions_snapshot"),
    ("sources",            "openalex.sources.openalex_sources_snapshot"),
    ("publishers",         "openalex.publishers.openalex_publishers_snapshot"),
    ("funders",            "openalex.funders.openalex_funders_snapshot"),
    ("awards",             "openalex.awards.openalex_awards_snapshot"),
    ("keywords",           "openalex.common.openalex_keywords_snapshot"),
    ("concepts",           "openalex.common.openalex_concepts_snapshot"),
    ("topics",             "openalex.common.openalex_topics_snapshot"),
    ("subfields",          "openalex.common.openalex_subfields_snapshot"),
    ("fields",             "openalex.common.openalex_fields_snapshot"),
    ("domains",            "openalex.common.openalex_domains_snapshot"),
    ("continents",         "openalex.common.openalex_continents_snapshot"),
    ("countries",          "openalex.common.openalex_countries_snapshot"),
    ("institution-types",  "openalex.common.openalex_institution_types_snapshot"),
    ("languages",          "openalex.common.openalex_languages_snapshot"),
    ("licenses",           "openalex.common.openalex_licenses_snapshot"),
    ("sdgs",               "openalex.common.openalex_sdgs_snapshot"),
    ("source-types",       "openalex.common.openalex_source_types_snapshot"),
    ("work-types",         "openalex.common.openalex_work_types_snapshot"),
]

print(f"  Checking manifests at: {s3_base}")

for entity_name, snapshot_table in MANIFEST_ENTITIES:
    manifest_path = f"{s3_base}/{entity_name}/manifest"

    # Check manifest exists and is readable
    try:
        manifest_raw = dbutils.fs.head(manifest_path, 10_000_000)  # up to 10MB
        manifest = json.loads(manifest_raw)
    except Exception as e:
        record(f"s3_manifest_{entity_name}", False, f"Manifest not found or unreadable: {e}")
        continue

    manifest_count = manifest.get("meta", {}).get("record_count", 0)
    manifest_size = manifest.get("meta", {}).get("content_length", 0)
    num_files = len(manifest.get("entries", []))

    # Manifest must have records
    if manifest_count == 0:
        record(f"s3_manifest_{entity_name}", False,
               f"Manifest exists but record_count=0 ({num_files} files, {manifest_size/(1024**2):.1f} MB)")
        continue

    # Cross-check against snapshot table count
    try:
        table_count = spark.read.table(snapshot_table).count()
        match = manifest_count == table_count
        detail = (f"manifest={manifest_count:,}  table={table_count:,}  "
                  f"files={num_files}  size={manifest_size/(1024**2):.1f} MB")
        if not match:
            detail += f"  MISMATCH (diff={abs(manifest_count - table_count):,})"
        record(f"s3_manifest_{entity_name}", match, detail)
    except Exception as e:
        # Table might not exist (e.g. if export task was skipped), just check manifest is non-empty
        record(f"s3_manifest_{entity_name}", True,
               f"manifest={manifest_count:,}  files={num_files}  size={manifest_size/(1024**2):.1f} MB (table check skipped: {e})")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("SMOKE TEST SUMMARY")
print("=" * 60)

passed_count = sum(1 for r in results if r["passed"])
failed_count = sum(1 for r in results if not r["passed"])
total_count = len(results)

print(f"\nTotal: {total_count} tests")
print(f"Passed: {passed_count}")
print(f"Failed: {failed_count}")

if failed_count > 0:
    print(f"\nFAILED TESTS:")
    for r in results:
        if not r["passed"]:
            print(f"  [XX] {r['test']}: {r['detail']}")
    print(f"\n{'!' * 60}")
    print(f"SNAPSHOT SMOKE TESTS FAILED - {failed_count} test(s) need attention")
    print(f"{'!' * 60}")
    # Raise to mark the Databricks task as failed
    raise Exception(f"Snapshot smoke tests failed: {failed_count}/{total_count} tests failed")
else:
    print(f"\nAll {total_count} tests passed. Snapshot looks good!")